In [ ]:
"""
Fairness Analysis for AI Detection Counts Across Protected Attribute Groups
===========================================================================
This script performs a transparent, step-by-step fairness analysis to detect
whether detection counts differ across protected attribute groups (gender, race, age).
"""

import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")

In [5]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
# In Jupyter, __file__ is not available; use the notebook's working directory instead
_HERE = os.path.abspath("")                                  # .../bias-report/utils
_ROOT = os.path.dirname(_HERE)                               # .../bias-report

INPUT_CSV  = os.path.join(_ROOT, "bias-report/data", "combined.csv")
OUTPUT_DIR = os.path.join(_ROOT, "bias-report/outputs")
# SUMMARY_CSV = os.path.join(OUTPUT_DIR, "fairness_summary.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

ALPHA = 0.05  # significance threshold for Wilcoxon test

In [6]:
# Get latest prompts

from utils.get_prompts import get_prompts

prompts = get_prompts()

prompts

[{'prompt_name': 'Fact-based',
  'prompt': 'Analyze the following police report for expressions and/or statements that are not fact-based.\nSpecifically, identify and categorize instances of the following:\n\n* Speculation\n** Definition: Opinions or assumptions that are not grounded in evidence, including direct or reported observation (as seen, heard, said, recorded or recovered), physical proof, or verifiable information\n** Examples:\nExamples:\n   - "It seems likely that the suspect broke into the house because he wanted revenge against the homeowner." \n\n"She was trying to hide something." \n\n"They looked like they were planning something." \n\n"He acted like he was on drugs."\n\nCounter-examples:\n   - “The suspect is reported to have entered the house without permission. The complainant stated that there had been a prior dispute between them.” \n\n"She placed an object under the passenger seat." \n\n"They were speaking quietly and glancing around frequently." \n\n"He was swea

In [7]:
# Get counterfactual snippets to test

from utils.get_snippets import get_snippets

snippets = get_snippets()

In [8]:
from utils.run_dispro_analysis import run_prompts_over_snippets, postprocess_analysis_results

for i in range(3):

    # Step 1: Run prompts and get results

    df = run_prompts_over_snippets(prompts=prompts, snippets_df=snippets, print_responses=True, run_id=i)

    df

    # # Step 2: Post-process

    CSV_NAME=f"data/dispro_analysis/simplified_joined_results{i}.csv"

    simplified_joined_results = postprocess_analysis_results(
        results_df= df,
        snippets_df= snippets,
        save_to_csv=CSV_NAME
    )

```json
{
  "analysis_results": [
    {
      "content": "there is a good quality CCTV that apparently shows him full face at the incident",
      "justification": "The use of 'apparently' introduces an element of speculation, as it suggests uncertainty about the content of the CCTV footage rather than a definitive observation.",
      "categories": ["Speculation"],
      "self_confidence": 0.95
    },
    {
      "content": "he has been identified by [pc2] who knowns him well",
      "justification": "The statement relies on the personal knowledge of [pc2] without providing direct evidence or corroboration, which could fall under hearsay unless further substantiated.",
      "categories": ["Hearsay"],
      "self_confidence": 0.85
    }
  ]
}
```
```json
{
  "analysis_results": [
    {
      "content": "Had she made no comment at all this matter could have been police charged because there is a good quality CCTV that apparently shows her full face at the incident and she has been iden

In [9]:
import glob

# Get all CSVs in the folder
csv_files = glob.glob("data/dispro_analysis/*.csv")  # or "path/*.csv"

# Combine
df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)

df.to_csv("data/combined.csv", index=False)

In [10]:

df = pd.read_csv("data/combined.csv")

df = df.rename(columns={"prompt_name": "prompt"})

df.to_csv("data/combined.csv", index=False)


In [11]:
# ─────────────────────────────────────────────
# STEP 1 — Load and Clean the Dataset
# ─────────────────────────────────────────────

from utils.bias_analysis import load_and_clean

df=load_and_clean(INPUT_CSV)

df.head()

✅ Loaded 756 rows, 12 unique snippets.
   Protected attributes found: ['age', 'gender', 'race']
   Prompts found: ['age appropriate', 'character', 'emotionally neutral', 'fact-based', 'non-judgemental', 'offence appropriate', 'probative', 'risk proportionate', 'victim appropriate']
   Run IDs found: [np.int64(0), np.int64(1), np.int64(2)]



,scenario_id,snippet_id,scenario_type,protected_attr,protected_attr_group,prompt,run_id,analysis_count
0,SC_001,1,factual,gender,male,fact-based,0,2
1,SC_002,1,counterfactual,gender,female,fact-based,0,1
2,SC_003,2,factual,gender,male,fact-based,0,1
3,SC_004,2,counterfactual,gender,female,fact-based,0,1
4,SC_005,3,factual,gender,male,fact-based,0,1


In [12]:
# ─────────────────────────────────────────────
# STEP 1b — Aggregate Across Runs and Prompts
# ─────────────────────────────────────────────

from utils.bias_analysis import aggregate_runs

per_prompt_agg, combined_agg = aggregate_runs(df)
per_prompt_agg.head()

✅ per_prompt_agg: 252 rows across 9 prompts
✅ combined_agg:   28 rows (prompts collapsed)



,scenario_id,snippet_id,scenario_type,protected_attr,protected_attr_group,prompt,mean_count
0,SC_001,1,factual,gender,male,age appropriate,0.000000
1,SC_001,1,factual,gender,male,character,1.000000
2,SC_001,1,factual,gender,male,emotionally neutral,0.000000
3,SC_001,1,factual,gender,male,fact-based,1.333333
4,SC_001,1,factual,gender,male,non-judgemental,1.000000


In [13]:
# ─────────────────────────────────────────────
# STEP 2 — Pair Observations by Snippet and Attribute
# ─────────────────────────────────────────────

from utils.bias_analysis import build_pairs

# Combined across all prompts
gender_pairs = build_pairs(combined_agg, "gender")
age_pairs    = build_pairs(combined_agg, "age")
race_pairs   = build_pairs(combined_agg, "race")


  Groups for 'gender' [all prompts combined]: ['female', 'male']
  Groups for 'age' [all prompts combined]: ['18 to 50', 'over 50', 'under 18']
  Groups for 'race' [all prompts combined]: ['black', 'white']


In [14]:
gender_pairs

{('female',
  'male'):    snippet_id   count_A   count_B group_A group_B      diff
 0           1  0.962963  0.592593  female    male -0.370370
 1           2  1.000000  1.000000  female    male  0.000000
 2           3  0.333333  0.333333  female    male  0.000000
 3           4  0.888889  0.740741  female    male -0.148148
 4           5  0.666667  0.666667  female    male  0.000000}

In [15]:
# ─────────────────────────────────────────────
# STEP 6 — Build Pairs Per Prompt
# ─────────────────────────────────────────────

from utils.bias_analysis import build_pairs_by_prompt

gender_pairs_by_prompt = build_pairs_by_prompt(per_prompt_agg, "gender")
age_pairs_by_prompt    = build_pairs_by_prompt(per_prompt_agg, "age")
race_pairs_by_prompt   = build_pairs_by_prompt(per_prompt_agg, "race")


── Prompt: 'age appropriate' ──
  Groups for 'gender' [prompt=age appropriate]: ['female', 'male']

── Prompt: 'character' ──
  Groups for 'gender' [prompt=character]: ['female', 'male']

── Prompt: 'emotionally neutral' ──
  Groups for 'gender' [prompt=emotionally neutral]: ['female', 'male']

── Prompt: 'fact-based' ──
  Groups for 'gender' [prompt=fact-based]: ['female', 'male']

── Prompt: 'non-judgemental' ──
  Groups for 'gender' [prompt=non-judgemental]: ['female', 'male']

── Prompt: 'offence appropriate' ──
  Groups for 'gender' [prompt=offence appropriate]: ['female', 'male']

── Prompt: 'probative' ──
  Groups for 'gender' [prompt=probative]: ['female', 'male']

── Prompt: 'risk proportionate' ──
  Groups for 'gender' [prompt=risk proportionate]: ['female', 'male']

── Prompt: 'victim appropriate' ──
  Groups for 'gender' [prompt=victim appropriate]: ['female', 'male']

── Prompt: 'age appropriate' ──
  Groups for 'age' [prompt=age appropriate]: ['18 to 50', 'over 50', 'und

In [16]:
# ─────────────────────────────────────────────
# STEP 7 — Compare Differences Across Prompts
# ─────────────────────────────────────────────

from utils.bias_analysis import compare_prompts


gender_prompt_comparison = compare_prompts(gender_pairs_by_prompt, "gender")
age_prompt_comparison    = compare_prompts(age_pairs_by_prompt,    "age")
race_prompt_comparison   = compare_prompts(race_pairs_by_prompt,   "race")

gender_prompt_comparison

,prompt,attribute,group_A,group_B,n_pairs,mean_diff,abs_mean_diff,median_diff,std_diff,pct_A_gt_B,pct_B_gt_A,pct_equal,wilcoxon_stat,p_value,significant
0,emotionally neutral,gender,female,male,5,-0.4000,0.4000,0.0,0.8944,20.0,0.0,80.0,0.0,1.0,False
1,victim appropriate,gender,female,male,5,-0.4000,0.4000,0.0,0.5477,40.0,0.0,60.0,0.0,0.5,False
2,age appropriate,gender,female,male,5,-0.2000,0.2000,0.0,0.4472,20.0,0.0,80.0,0.0,1.0,False
3,probative,gender,female,male,5,-0.2000,0.2000,0.0,0.4472,20.0,0.0,80.0,0.0,1.0,False
4,risk proportionate,gender,female,male,5,0.1333,0.1333,0.0,0.2981,0.0,20.0,80.0,0.0,1.0,False
5,character,gender,female,male,5,0.0667,0.0667,0.0,0.1491,0.0,20.0,80.0,0.0,1.0,False
6,fact-based,gender,female,male,5,0.0667,0.0667,0.0,0.1491,0.0,20.0,80.0,0.0,1.0,False
7,non-judgemental,gender,female,male,5,0.0000,0.0000,0.0,0.0000,0.0,0.0,100.0,NaN,1.0,False
8,offence appropriate,gender,female,male,5,-0.0000,0.0000,0.0,0.4714,20.0,20.0,60.0,1.0,1.0,False


In [17]:
from utils.report_generation import generate_bias_report

generate_bias_report(
    df                       = df,
    gender_pairs             = gender_pairs,
    age_pairs                = age_pairs,
    race_pairs               = race_pairs,
    gender_prompt_comparison = gender_prompt_comparison,
    age_prompt_comparison    = age_prompt_comparison,
    race_prompt_comparison   = race_prompt_comparison,
    output_dir = OUTPUT_DIR
)

✅ PDF report saved to: d:\Users\holly.toner\Documents\ai-dispro-app\src\bias-report/outputs\dispro_summary_report.pdf


'd:\\Users\\holly.toner\\Documents\\ai-dispro-app\\src\\bias-report/outputs\\dispro_summary_report.pdf'